### Layer 1 — The Money Trail
*Tables: contracts + countries*

> When a developing country gets a World Bank loan, does the money stay
> in the country or flow straight back out to foreign suppliers?

**Questions:**
1. What percentage of total contract value goes to **local vs foreign suppliers**?
2. Which **supplier countries** benefit the most from World Bank contracts?
3. Do **low income countries** get more foreign suppliers than high income ones?
4. Which **sectors** (education, infrastructure, health) attract the most foreign suppliers?
5. Is there a pattern between a country's **income level** and where its suppliers come from?


In [2]:
import pandas as pd

contracts = pd.read_csv("../data/cleaned/contracts_clean.csv")
countries = pd.read_csv("../data/cleaned/countries_clean.csv")

print(f"contracts: {contracts.shape}")
print(f"countries: {countries.shape}")

contracts: (273497, 20)
countries: (211, 10)


What percentage of total contract value goes to local vs foreign suppliers?

In [6]:
contracts_enriched = contracts.merge(
    countries[["country_code_iso2", "country_code_iso3", "income_level", "lending_type"]],
    left_on="borrower_country_code",
    right_on="country_code_iso2",
    how="left"
)

print(f"Shape after join: {contracts_enriched.shape}")
print(f"Unmatched rows: {contracts_enriched['income_level'].isna().sum()}")

Shape after join: (273497, 24)
Unmatched rows: 40731


In [7]:
contracts_enriched["supplier_type"] = contracts_enriched.apply(
    lambda row: "Local" if row["borrower_country_code"] == row["supplier_country_code"] 
    else "Foreign", axis=1
)

money_flow = contracts_enriched.groupby("supplier_type")["supplier_contract_amount_usd"].sum()
money_flow_pct = (money_flow / money_flow.sum() * 100).round(2)

print("Total contract value (USD):")
print(money_flow.apply(lambda x: f"${x:,.0f}"))
print(f"\nPercentage split:")
print(money_flow_pct.apply(lambda x: f"{x}%"))

Total contract value (USD):
supplier_type
Foreign    $54,459,310,245
Local      $67,701,721,948
Name: supplier_contract_amount_usd, dtype: object

Percentage split:
supplier_type
Foreign    44.58%
Local      55.42%
Name: supplier_contract_amount_usd, dtype: object


Which supplier countries benefit the most from World Bank contracts?

In [51]:
win_countries = contracts_enriched[
    ~contracts_enriched["supplier_country"].isin(["World", "Commonwealth of Independent States"])
].groupby("supplier_country").agg(
    total_value_in_USD=("supplier_contract_amount_usd", "sum"),
    contract_count=("wb_contract_number", "count")
).sort_values("total_value_in_USD", ascending=False)

print(win_countries.head(10))

                  total_value_in_USD  contract_count
supplier_country                                    
China                   2.108345e+10            4001
India                   1.020583e+10           35029
Turkiye                 6.388603e+09            3766
Pakistan                3.510101e+09            5460
Nigeria                 3.370149e+09            8939
Bangladesh              2.743712e+09            4705
Viet Nam                2.468121e+09            3403
Indonesia               2.331680e+09            5212
Ethiopia                2.030539e+09           12615
Brazil                  1.720418e+09            3498


Do low income countries get more foreign suppliers than high income ones?

In [36]:
contracts_enriched["supplier_type"] = contracts_enriched.apply(
    lambda row: "Local" if row["borrower_country_code"] == row["supplier_country_code"] 
    else "Foreign", axis=1
)
income = contracts_enriched.groupby(
    ["income_level", "supplier_type"]
)["supplier_contract_amount_usd"].sum().reset_index()

income["pct"] = income.groupby("income_level")["supplier_contract_amount_usd"].transform(
    lambda x: (x / x.sum() * 100).round(2)
)

print(income.sort_values(["income_level", "supplier_type"]).to_string())

          income_level supplier_type  supplier_contract_amount_usd    pct
0          High income       Foreign                  7.123699e+08  22.69
1          High income         Local                  2.427130e+09  77.31
2           Low income       Foreign                  8.718163e+09  53.57
3           Low income         Local                  7.554741e+09  46.43
4  Lower middle income       Foreign                  2.099725e+10  38.48
5  Lower middle income         Local                  3.356445e+10  61.52
6       Not classified       Foreign                  1.935374e+09  55.10
7       Not classified         Local                  1.577015e+09  44.90
8  Upper middle income       Foreign                  8.543797e+09  28.89
9  Upper middle income         Local                  2.103149e+10  71.11


Which sectors (education, infrastructure, health) attract the most foreign suppliers?

In [41]:
sectors_exploded = contracts_enriched.copy()
sectors_exploded["project_global_practice"] = sectors_exploded["project_global_practice"].str.split(";")
sectors_exploded = sectors_exploded.explode("project_global_practice")
sectors_exploded["project_global_practice"] = sectors_exploded["project_global_practice"].str.strip()

sectors = sectors_exploded.groupby(
    ["project_global_practice", "supplier_type"]
)["supplier_contract_amount_usd"].sum().reset_index()

sectors["pct"] = sectors.groupby("project_global_practice")["supplier_contract_amount_usd"].transform(
    lambda x: (x / x.sum() * 100).round(2)
)

foreign_sectors = sectors[sectors["supplier_type"] == "Foreign"].sort_values("pct", ascending=False)
print(foreign_sectors[["project_global_practice", "supplier_contract_amount_usd", "pct"]].to_string())

   project_global_practice  supplier_contract_amount_usd     pct
2      Digital Development                  2.000000e+04  100.00
5     Energy & Extractives                  1.202110e+10   71.38
11                  Health                  1.010173e+10   53.02
7       Energy and Mineral                  3.014253e+05   50.04
21       Social Protection                  8.479917e+09   46.93
17    Info & Communication                  5.025835e+09   46.83
13    Industry & Trade/Ser                  7.148990e+09   42.37
25          Transportation                  1.337672e+10   41.49
27                 Unknown                  4.072750e+09   35.68
0              Agriculture                  8.341049e+09   35.48
30       Water/Sanit/Waste                  9.077836e+09   34.36
23          Social Sustain                  4.964217e+06   34.21
19            Public Admin                  6.745950e+09   30.43
9         Financial Sector                  6.555453e+08   24.26
3                Educatio

Is there a pattern between a country's income level and where its suppliers come from?

In [45]:
exclude = ["World", "Commonwealth of Independent States"]

pattern = contracts_enriched[
    ~contracts_enriched["supplier_country"].isin(exclude)
].groupby(
    ["income_level", "supplier_country"]
)["supplier_contract_amount_usd"].sum().reset_index()

top3 = pattern.groupby("income_level").apply(
    lambda x: x.nlargest(3, "supplier_contract_amount_usd")
).reset_index(drop=True)

print(top3.to_string())

           income_level      supplier_country  supplier_contract_amount_usd
0           High income                Poland                  9.418702e+08
1           High income               Romania                  7.479208e+08
2           High income               Croatia                  2.941637e+08
3            Low income                 China                  2.849611e+09
4            Low income          Burkina Faso                  1.330596e+09
5            Low income            Mozambique                  1.053928e+09
6   Lower middle income                 China                  9.382347e+09
7   Lower middle income                 India                  8.828582e+09
8   Lower middle income              Pakistan                  3.496943e+09
9        Not classified              Ethiopia                  1.577015e+09
10       Not classified                 China                  9.967715e+08
11       Not classified  United Arab Emirates                  1.715021e+08
12  Upper mi

C:\Users\Fatima zahra\AppData\Local\Temp\ipykernel_21552\298632550.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top3 = pattern.groupby("income_level").apply(


## Layer 1: The Money Trail: Findings Summary

### Core Question
> When a developing country gets a World Bank loan, does the money 
> stay in the country or flow straight back out to foreign suppliers?


#### Q1: Local vs Foreign: The Overall Split

**Finding:**
| Supplier Type | Total Value | Percentage |
|---|---|---|
| Local | 67.7BUSD | 55.42% |
| Foreign | 54.5BUSD | 44.58% |

On the surface this looks positive — more than half stays local.
But this is an average across ALL countries and hides a much darker
pattern underneath.


#### Q2: Which Countries Capture the Most Contract Money?

**Finding — Top 5 Supplier Countries:**
| Country | Total Value | Contract Count |
|---|---|---|
| China | 21.1BUSD | 4,001 |
| India | 10.2BUSD | 35,029 |
| Turkey | 6.4BUSD | 3,766 |
| Pakistan | 3.5BUSD | 5,460 |
| Nigeria | 3.4BUSD | 8,939 |

**Key insight:** The biggest winners are not Western nations — they are
large, powerful developing countries themselves. China alone captured
21BUSD, nearly double India in second place. Western countries don't
even appear in the top 10.

This reframes the question — it's not rich vs poor, it's
**powerful developing countries extracting from weaker ones.**


#### Q3: Does Income Level Change the Story?

**Finding — Foreign supplier % by income group:**
| Income Level | Foreign % | Local % |
|---|---|---|
| High income | 22.69% | 77.31% |
| Upper middle income | 28.89% | 71.11% |
| Lower middle income | 38.48% | 61.52% |
| Low income | **53.57%** | **46.43%** |

**Key insight:** This is the most important finding in Layer 1.

> The poorer the country, the more of its loan money 
> goes to foreign suppliers.

Low income countries are the only group where **foreign suppliers
capture the majority** of contract value. Rich countries keep 77%
locally — low income countries lose more than half immediately.

**Why this happens:**
- Lack of local engineering and consulting capacity
- World Bank open bidding rules favor large experienced foreign firms
- Chinese and Indian firms aggressively underbid local companies


#### Q4: Which Sectors Bleed the Most to Foreign Suppliers?

**Finding — Foreign % by sector:**
| Sector | Foreign % | Interpretation |
|---|---|---|
| Energy & Extractives | 71.38% | Specialized tech — rarely available locally |
| Health | 53.02% | Medical equipment — mostly imported |
| Social Protection | 46.93% | Consulting heavy — foreign firms dominate |
| Info & Communication | 46.83% | Tech infrastructure — foreign tech wins |
| Transportation | 41.49% | Large infrastructure — Chinese firms dominate |
| **Education** | **22.91%** | People-based — stays local |
| **Public Admin** | **30.43%** | Community-based — stays local |

**Key insight:** Sectors requiring specialized technology and equipment
bleed the most to foreign suppliers. When a poor country borrows for
an energy project, **71 cents of every dollar leaves the country
immediately.**

Education and Public Admin are the rare bright spots — contracts
tend to stay within the borrowing country because the work is
people and community based.


#### Q5: Is There a Pattern Between Income Level and Supplier Origin?

**Finding — Top 3 suppliers per income group:**
| Income Group | #1 | #2 | #3 |
|---|---|---|---|
| High income | Poland | Romania | Croatia |
| Upper middle | Turkey | China | Indonesia |
| Lower middle | China | India | Pakistan |
| Low income | **China** | Burkina Faso | Mozambique |

**Key insight:** China appears as the dominant supplier across every
income group except high income. The pattern is stark:

- **High income** countries hire regional neighbors — money stays
  within their economic orbit
- **Middle income** countries are dominated by China and India —
  the world's contract powerhouses
- **Low income** countries are China's biggest opportunity —
  2.8BUSD extracted from the world's poorest borrowers


### Layer 1: Overall Verdict

The 55% local average from Q1 is misleading. The real story is:

1. **Low income countries lose the majority of their loan money
   to foreign suppliers** — the ones who need it most benefit the least
2. **China is the single biggest winner** of World Bank contract money
   across all developing country income groups
3. **Energy and infrastructure loans** are the worst offenders —
   nearly 3/4 of the money leaves immediately
4. **Education loans** are the exception — money tends to stay local

> A low income country borrowing for an energy project is essentially
> taking on debt to pay Chinese companies — while its own economy
> sees little of the money it borrowed.

That is the Money Trail.